In [ ]:
from functools import lru_cache
from pathlib import Path
import time

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset

# DGCNN for MARS

Notebook version of `src/SK-DGCNN/dgcnn/pytorch/model.py` adapted to the cleaned MARS radar folders.

The original project classifies one point cloud as `(B, C, N)`. Here, one sample is a 60-frame radar window collapsed into a fixed-size point cloud, then passed to DGCNN with the same EdgeConv block pattern:

`EdgeConv 64 -> EdgeConv 64 -> EdgeConv 128 -> EdgeConv 256 -> 1024 embedding -> max/avg pooling -> 512 -> 256 -> classes`

MARS raw files store `radar_data_cropped` with shape `(8, total_points)`. Row 0 is the frame id. Rows `(2, 3, 4, 5, 6)` are used as the model channels: `x, y, z, velocity, intensity`.

Dimension convention:

- B: batch size
- C: feature channels, set to 5
- N: point count per DGCNN sample, set to 512
- T: source frames per window, set to 60 before the window is flattened into N points

In [ ]:
PROJECT_ROOT = Path('/Users/elo/Coding/radar_ml')
DATA_ROOT = PROJECT_ROOT / 'data' / 'MARS'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / 'MARS_DGCNN'

ACTIVITY_CLASS_NAMES = (
    'Left_upper_limb_extension',
    'Right_upper_limb_extension',
    'Both_upper_limb_extension',
    'Left_front_lunge',
    'Right_front_lunge',
    'Squad',
    'Left_side_lunge',
    'Right_side_lunge',
    'Left_limb_extension',
    'Right_limb_extension',
)
ACTIVITY_INDEX = {name: idx for idx, name in enumerate(ACTIVITY_CLASS_NAMES)}

DEVICE = 'mps'
CONFIG = {
    'mat_key': 'radar_data_cropped',
    'frame_row': 0,
    'feature_names': ('x', 'y', 'z', 'velocity', 'intensity'),
    'feature_rows': (2, 3, 4, 5, 6),
    'feature_amount': 5,
    'frame_count': 60,
    'stride_frame_count': 60,
    'num_points': 512,
    'validation_subject': 'subject1',
    'test_subject': 'subject3',
    'k': 20,
    'emb_dims': 1024,
    'dropout': 0.5,
    'batch_size': 16,
    'test_batch_size': 16,
    'epochs': 20,
    'lr': 0.001,
    'momentum': 0.9,
    'use_sgd': True,
}

In [ ]:
print(f'Classes: {len(ACTIVITY_CLASS_NAMES)}')
print(f'Window before DGCNN: T={CONFIG["frame_count"]}, raw D={CONFIG["feature_amount"]}')
print(f'DGCNN input shape: (B, C={CONFIG["feature_amount"]}, N={CONFIG["num_points"]})')
print(dict(zip(CONFIG['feature_names'], CONFIG['feature_rows'])))

# Data

In [ ]:
def mars_file_paths():
    "Return all MARS radar files in class-folder order."
    return [
        file_path
        for class_name in ACTIVITY_CLASS_NAMES
        for file_path in sorted((DATA_ROOT / class_name).glob('subject*/radar_data*.mat'))
    ]

In [ ]:
def split_file_paths():
    "Split files by subject."
    train_files = []
    val_files = []
    test_files = []
    for file_path in mars_file_paths():
        subject = file_path.parent.name
        if subject == CONFIG['test_subject']:
            test_files.append(str(file_path))
        elif subject == CONFIG['validation_subject']:
            val_files.append(str(file_path))
        else:
            train_files.append(str(file_path))
    return train_files, val_files, test_files

In [ ]:
@lru_cache(maxsize=None)
def load_mars_file_raw(file_path):
    "Load one MARS file and return frames as raw point tensors."
    with h5py.File(file_path, 'r') as handle:
        radar = np.asarray(handle[CONFIG['mat_key']])
    frame_ids = radar[CONFIG['frame_row']].astype(np.int64)
    features = radar[list(CONFIG['feature_rows'])].T.astype(np.float32)
    return [torch.from_numpy(features[frame_ids == frame_id]) for frame_id in np.unique(frame_ids)]

In [ ]:
@lru_cache(maxsize=None)
def train_normalization_stats():
    "Compute normalization stats over training subjects."
    train_files, _, _ = split_file_paths()
    points = [torch.cat(load_mars_file_raw(file_path), dim=0) for file_path in train_files]
    points = torch.cat(points, dim=0)
    return points.mean(dim=0), points.std(dim=0).clamp_min(1e-6)

In [ ]:
@lru_cache(maxsize=None)
def load_mars_file(file_path):
    "Load one MARS file and return normalized frames."
    mean, std = train_normalization_stats()
    return [(frame - mean) / std for frame in load_mars_file_raw(file_path)]

In [ ]:
def build_window_index(file_paths):
    "Create labeled 60-frame window indexes."
    samples = []
    for file_path in file_paths:
        class_name = Path(file_path).parents[1].name
        frames = load_mars_file(file_path)
        max_start = len(frames) - CONFIG['frame_count']
        for start_idx in range(0, max_start + 1, CONFIG['stride_frame_count']):
            samples.append((file_path, start_idx, ACTIVITY_INDEX[class_name]))
    return samples

In [ ]:
def process_window(frames):
    "Collapse a frame window into one fixed-size DGCNN point cloud."
    points = torch.cat(frames, dim=0)
    return points[:CONFIG['num_points']]

In [ ]:
class MARSDGCNNDataset(Dataset):
    "Dataset returning one DGCNN point cloud and one movement label."
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, start_idx, label_idx = self.samples[idx]
        frames = load_mars_file(file_path)
        window = frames[start_idx:start_idx + CONFIG['frame_count']]
        point_cloud = process_window(window)
        label = torch.tensor(label_idx, dtype=torch.long)
        return point_cloud, label

In [ ]:
train_files, valid_files, test_files = split_file_paths()
train_samples = build_window_index(train_files)
valid_samples = build_window_index(valid_files)
test_samples = build_window_index(test_files)
print(len(train_files), len(valid_files), len(test_files))
print(len(train_samples), len(valid_samples), len(test_samples))
print(train_samples[:3])

In [ ]:
train_data = MARSDGCNNDataset(train_samples)
x, y = train_data[0]
print(x.shape, y)

# Architecture

In [ ]:
def knn(x, k):
    "K nearest neighbor indexes from the original DGCNN implementation."
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    return pairwise_distance.topk(k=k, dim=-1)[1]

In [ ]:
def get_graph_feature(x, k=20, idx=None):
    "Build EdgeConv features as concat(neighbor - center, center)."
    batch_size = x.size(0)
    num_points = x.size(2)
    x = x.view(batch_size, -1, num_points)
    if idx is None:
        idx = knn(x, k=k)
    idx_base = torch.arange(0, batch_size, device=x.device).view(-1, 1, 1) * num_points
    idx = (idx + idx_base).view(-1)
    _, num_dims, _ = x.size()
    x = x.transpose(2, 1).contiguous()
    feature = x.view(batch_size * num_points, -1)[idx, :]
    feature = feature.view(batch_size, num_points, k, num_dims)
    x = x.view(batch_size, num_points, 1, num_dims).repeat(1, 1, k, 1)
    return torch.cat((feature - x, x), dim=3).permute(0, 3, 1, 2).contiguous()

In [ ]:
class DGCNN(nn.Module):
    "DGCNN architecture from `src/SK-DGCNN/dgcnn/pytorch/model.py`."
    def __init__(self, output_channels=len(ACTIVITY_CLASS_NAMES)):
        super().__init__()
        self.k = CONFIG['k']
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)
        self.bn5 = nn.BatchNorm1d(CONFIG['emb_dims'])
        self.conv1 = nn.Sequential(nn.Conv2d(CONFIG['feature_amount'] * 2, 64, kernel_size=1, bias=False), self.bn1, nn.LeakyReLU(negative_slope=0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(64 * 2, 64, kernel_size=1, bias=False), self.bn2, nn.LeakyReLU(negative_slope=0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(64 * 2, 128, kernel_size=1, bias=False), self.bn3, nn.LeakyReLU(negative_slope=0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(128 * 2, 256, kernel_size=1, bias=False), self.bn4, nn.LeakyReLU(negative_slope=0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(512, CONFIG['emb_dims'], kernel_size=1, bias=False), self.bn5, nn.LeakyReLU(negative_slope=0.2))
        self.linear1 = nn.Linear(CONFIG['emb_dims'] * 2, 512, bias=False)
        self.bn6 = nn.BatchNorm1d(512)
        self.dp1 = nn.Dropout(p=CONFIG['dropout'])
        self.linear2 = nn.Linear(512, 256)
        self.bn7 = nn.BatchNorm1d(256)
        self.dp2 = nn.Dropout(p=CONFIG['dropout'])
        self.linear3 = nn.Linear(256, output_channels)

    def forward(self, x):
        batch_size = x.size(0)
        x = get_graph_feature(x, k=self.k)
        x = self.conv1(x)
        x1 = x.max(dim=-1, keepdim=False)[0]
        x = get_graph_feature(x1, k=self.k)
        x = self.conv2(x)
        x2 = x.max(dim=-1, keepdim=False)[0]
        x = get_graph_feature(x2, k=self.k)
        x = self.conv3(x)
        x3 = x.max(dim=-1, keepdim=False)[0]
        x = get_graph_feature(x3, k=self.k)
        x = self.conv4(x)
        x4 = x.max(dim=-1, keepdim=False)[0]
        x = torch.cat((x1, x2, x3, x4), dim=1)
        x = self.conv5(x)
        x1 = F.adaptive_max_pool1d(x, 1).view(batch_size, -1)
        x2 = F.adaptive_avg_pool1d(x, 1).view(batch_size, -1)
        x = torch.cat((x1, x2), 1)
        x = F.leaky_relu(self.bn6(self.linear1(x)), negative_slope=0.2)
        x = self.dp1(x)
        x = F.leaky_relu(self.bn7(self.linear2(x)), negative_slope=0.2)
        x = self.dp2(x)
        return self.linear3(x)

In [ ]:
model = DGCNN()
train_loader = DataLoader(train_data, batch_size=CONFIG['batch_size'], shuffle=True, drop_last=True)
sample_batch, sample_labels = next(iter(train_loader))
logits = model(sample_batch.permute(0, 2, 1))
print(sample_batch.shape)
print(sample_batch.permute(0, 2, 1).shape)
print(logits.shape, sample_labels.shape)

# Training

In [ ]:
def cal_loss(pred, gold, smoothing=True):
    "Cross entropy with the label smoothing used by the DGCNN project."
    gold = gold.contiguous().view(-1)
    if smoothing:
        eps = 0.2
        n_class = pred.size(1)
        one_hot = torch.zeros_like(pred).scatter(1, gold.view(-1, 1), 1)
        one_hot = one_hot * (1 - eps) + (1 - one_hot) * eps / (n_class - 1)
        log_prb = F.log_softmax(pred, dim=1)
        return -(one_hot * log_prb).sum(dim=1).mean()
    return F.cross_entropy(pred, gold, reduction='mean')

In [ ]:
def accuracy(predictions, labels):
    "Compute standard accuracy."
    return (predictions == labels).float().mean().item()

In [ ]:
def balanced_accuracy(predictions, labels):
    "Compute mean per-class recall."
    scores = [(predictions[labels == idx] == idx).float().mean() for idx in range(len(ACTIVITY_CLASS_NAMES))]
    return torch.stack(scores).mean().item()

In [ ]:
def create_dataloaders():
    "Create train, validation, and test dataloaders."
    train_loader = DataLoader(MARSDGCNNDataset(train_samples), batch_size=CONFIG['batch_size'], shuffle=True, drop_last=True)
    valid_loader = DataLoader(MARSDGCNNDataset(valid_samples), batch_size=CONFIG['test_batch_size'], shuffle=False, drop_last=False)
    test_loader = DataLoader(MARSDGCNNDataset(test_samples), batch_size=CONFIG['test_batch_size'], shuffle=False, drop_last=False)
    return train_loader, valid_loader, test_loader

In [ ]:
def run_epoch(model, dataloader, optimizer=None):
    "Run one training or evaluation epoch."
    model.train(optimizer is not None)
    total_loss = 0.0
    total_count = 0
    all_predictions = []
    all_labels = []
    for data, label in dataloader:
        data = data.to(DEVICE).permute(0, 2, 1)
        label = label.to(DEVICE).squeeze()
        if optimizer is not None:
            optimizer.zero_grad()
        logits = model(data)
        loss = cal_loss(logits, label)
        if optimizer is not None:
            loss.backward()
            optimizer.step()
        predictions = logits.max(dim=1)[1]
        total_loss += loss.item() * data.size(0)
        total_count += data.size(0)
        all_predictions.append(predictions.detach().cpu())
        all_labels.append(label.detach().cpu())
    all_predictions = torch.cat(all_predictions)
    all_labels = torch.cat(all_labels)
    return total_loss / total_count, accuracy(all_predictions, all_labels), balanced_accuracy(all_predictions, all_labels)

In [ ]:
def train_model():
    "Train DGCNN with the optimizer and scheduler from the project."
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
    model = DGCNN().to(DEVICE)
    train_loader, valid_loader, _ = create_dataloaders()
    if CONFIG['use_sgd']:
        optimizer = torch.optim.SGD(model.parameters(), lr=CONFIG['lr'] * 100, momentum=CONFIG['momentum'], weight_decay=1e-4)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, CONFIG['epochs'], eta_min=CONFIG['lr'])
    best_valid_acc = 0.0
    for epoch in range(CONFIG['epochs']):
        scheduler.step()
        start = time.time()
        train_loss, train_acc, train_balanced_acc = run_epoch(model, train_loader, optimizer)
        with torch.no_grad():
            valid_loss, valid_acc, valid_balanced_acc = run_epoch(model, valid_loader)
        print(f'Epoch {epoch}: train loss {train_loss:.6f}, train acc {train_acc:.6f}, train avg acc {train_balanced_acc:.6f}')
        print(f'Epoch {epoch}: valid loss {valid_loss:.6f}, valid acc {valid_acc:.6f}, valid avg acc {valid_balanced_acc:.6f}, time {time.time() - start:.1f}s')
        if valid_acc >= best_valid_acc:
            best_valid_acc = valid_acc
            torch.save(model.state_dict(), CHECKPOINT_ROOT / 'model.pt')
    return model

In [ ]:
# model = train_model()

# Inference

In [ ]:
def evaluate_checkpoint(checkpoint_path):
    "Evaluate a saved DGCNN checkpoint on the held-out subject."
    model = DGCNN().to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()
    _, _, test_loader = create_dataloaders()
    with torch.no_grad():
        loss, acc, avg_acc = run_epoch(model, test_loader)
    return loss, acc, avg_acc

In [ ]:
# loss, acc, avg_acc = evaluate_checkpoint(CHECKPOINT_ROOT / 'model.pt')
# print(f'Test loss {loss:.6f}, test acc {acc:.6f}, test avg acc {avg_acc:.6f}')